## DT-215


In [0]:
%sql
ALTER TABLE `p-cdh`.primaryactive.party 
ADD COLUMN RiskDescription string 
AFTER OrganizationOriginCountry;

In [0]:
%sql
ALTER TABLE `p-cdh`.primaryarchive.party 
ADD COLUMN RiskDescription string 
AFTER OrganizationOriginCountry;

In [0]:
%sql
CREATE OR REPLACE VIEW `p-cdh`.primary.party AS 
SELECT * FROM `p-cdh`.primaryactive.party;

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativeactive.dparty 
ADD COLUMN RiskDescription string 
AFTER PartyPostalCode;

In [0]:
%sql
ALTER TABLE `p-cdh`.derivativearchive.dparty 
ADD COLUMN RiskDescription string 
AFTER PartyPostalCode;

In [0]:
%sql
CREATE OR REPLACE VIEW `p-cdh`.derivative.dparty as 
SELECT * FROM `p-cdh`.derivativearchive.dparty 
WHERE BusinessDate < (SELECT MIN(BusinessDate) as start_date FROM `p-cdh`.derivativeactive.dparty) 
UNION ALL 
SELECT * FROM `p-cdh`.derivativeactive.dparty;

## DATA2-270

In [0]:
%sql
create or replace table `p-cdh`.reference.qca_dim_customer_addresses_extra as 
select * from `d-cdh`.reference.qca_dim_customer_addresses_extra;

###Refresh views

In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('p-cdh')
      AND table_schema in ('derivative', 'primary')
      and table_type='VIEW'
""")

# display(views_df)

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    # print(table_name)

    full_view_name =f"`{table_catalog}`.{table_schema}.{table_name}"

    describe_full_view_name = spark.sql(f"""
              DESCRIBE EXTENDED {full_view_name}""")
    
    describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

   # print(describe_full_view_name)

    describe_query = spark.sql(f"""
              select data_type--REGEXP_REPLACE(TRIM(data_type), '[^a-zA-Z0-9*<> ]', '') 
              AS cleaned_view_text 
              from describe_full_view_name 
              where col_name ='View Text'
              """)
    
    sql_view_creation = describe_query.collect()[0][0]

    create_or_replace_sql = f"""
          create or replace view {full_view_name} as 
          {sql_view_creation}
          """
    
    spark.sql(create_or_replace_sql)



In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativearchive','primaryarchive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


In [0]:
views_df = spark.sql("""
    SELECT table_name, table_schema, table_catalog
    FROM system.information_schema.tables
    WHERE table_catalog in ('qa-cdh')
      AND table_schema in ('derivativeactive','primaryactive')
      AND table_type='VIEW'
""")

for view_row in views_df.collect():
    table_name = view_row["table_name"]
    table_schema = view_row["table_schema"]
    table_catalog = view_row["table_catalog"]

    full_view_name = f"`{table_catalog}`.{table_schema}.{table_name}"

    try:
        describe_full_view_name = spark.sql(f"""
            DESCRIBE EXTENDED {full_view_name}
        """)

        describe_full_view_name.createOrReplaceTempView("describe_full_view_name")

        describe_query = spark.sql("""
            SELECT data_type
            FROM describe_full_view_name 
            WHERE col_name = 'View Text'
        """)

        sql_view_creation = describe_query.collect()[0][0]

        create_or_replace_sql = f"""
            CREATE OR REPLACE VIEW {full_view_name} AS 
            {sql_view_creation}
        """

        spark.sql(create_or_replace_sql)

        print(f"Successfully refreshed view: {full_view_name}")

    except Exception as e:
        print(f"   Failed to refresh view: {full_view_name}")


###Closed loans function update

In [0]:
env='p'
# Get the schema of the dexposure table
dexposure_table = spark.table(f"`{env}-cdh`.derivative.dexposure")
dexposure_schema_fields = dexposure_table.schema.fields


# Convert the schema fields into the required string format
dexposure_schema = ",\n".join([
    f"{field.name} {field.dataType.simpleString()}"
    for field in dexposure_schema_fields
])


spark.sql(f'''
CREATE OR REPLACE FUNCTION `{env}-cdh`.derivative.get_all_exposures_by_business_date(
    business_date Date
)
RETURNS TABLE 
( 
{dexposure_schema}
)
RETURN

SELECT B.*
FROM(
        SELECT  ExposureKey, MAX(BusinessDate) BusinessDate
        FROM `p-cdh`.derivative.dexposure
        WHERE BusinessDate <= business_date
        GROUP BY ExposureKey
) A
INNER JOIN   `{env}-cdh`.derivative.dexposure B
on a.ExposureKey = b.ExposureKey
and a.BusinessDate = b.BusinessDate
AND B.IsActiveFlagInt = 0
AND coalesce(B.JustificationCode,0) != 43023

UNION

SELECT * FROM `{env}-cdh`.derivative.dexposure
WHERE BusinessDate = business_date
AND IsActiveFlagInt = 1
''')